# Data Acquisition
Downloads raw data from three sources via - **OpenDOSM, MoH Malaysia GitHub, World Bank API**

All raw files are saved to `data/raw/`. Re-run with `force=True` to re-download existing files.

In [1]:
import sys
from pathlib import Path

# Ensure project root is on sys.path so the scripts package is importable
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    
from scripts.acquire import run
from scripts.config import RAW

## Download

Set  to re-download all files even if they already exist on disk.

In [2]:
run(force=False)


--- OpenDOSM ---
  [skip] hh_income_parlimen.csv already exists
  [skip] hh_poverty_parlimen.csv already exists
  [skip] hh_income_state.csv already exists
  [skip] hh_poverty_state.csv already exists
  [skip] population_state.csv already exists
  [skip] hh_inequality_state.csv already exists
  [skip] cpi_national.csv already exists

--- MoH Malaysia ---
  [skip] moh_facilities.csv already exists
  [skip] moh_beds.csv already exists

--- World Bank ---
  [skip] worldbank_malaysia.csv already exists

✓ Raw data in C:\Users\aliff\Documents\GitHub\inequality-analysis\data\raw


## Inventory Check

Confirm all expected files are present and non-empty.

In [3]:
import pandas as pd

EXPECTED = [
    "hh_income_parlimen.csv",
    "hh_poverty_parlimen.csv",
    "hh_income_state.csv",
    "hh_poverty_state.csv",
    "hh_inequality_state.csv",
    "population_state.csv",
    "cpi_state.csv",
    "moh_facilities.csv",
    "moh_beds.csv",
    "worldbank_malaysia.csv",
]

print(f"{'File':<30} {'Rows':>6}  {'Cols':>4}  Status")
print("-" * 60)
for fname in EXPECTED:
    path = RAW / fname
    if path.exists():
        df = pd.read_csv(path)
        print(f"{fname:<30} {len(df):>6}  {df.shape[1]:>4}  OK")
    else:
        print(f"{fname:<30} {'':>6}  {'':>4}  MISSING")

File                             Rows  Cols  Status
------------------------------------------------------------
hh_income_parlimen.csv            666     5  OK
hh_poverty_parlimen.csv           666     4  OK
hh_income_state.csv               303     4  OK
hh_poverty_state.csv              294     5  OK
hh_inequality_state.csv           273     3  OK


population_state.csv           263679     6  OK


cpi_state.csv                     416    18  OK
moh_facilities.csv               3304     9  OK
moh_beds.csv                      149     8  OK
worldbank_malaysia.csv             20     5  OK


## Notes on Data Scope

- **Parliament-level granularity**: `hh_income_parlimen` and `hh_poverty_parlimen` cover survey years 2019, 2022, and 2024 at parliament constituency level. These are aggregated to state level in ETL.
- **Historical state-level series**: `hh_income_state` and `hh_poverty_state` span 1970-2022 at state level (official DOSM HIES aggregates).
- **Official state Gini**: `hh_inequality_state` provides DOSM's household-level Gini coefficient by state for survey years 1974–2022. This replaces the inter-constituency proxy Gini derived from parliament data in `combined_state`.
- **State-level CPI**: `cpi_state` provides monthly CPI by state and category (base 2010=100). ETL extracts the `overall` category and averages to annual. Coverage depends on the DOSM snapshot (typically 2021–present). Used to compute real income where year overlap exists.
- **Health infrastructure snapshot**: MoH files are a single point-in-time snapshot with no year column. Infrastructure counts are static; only per-capita metrics vary by year as population denominators change.
- **World Bank data is national-level only** and is not used for state-level analysis.